<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/treatmentsteps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Treatment design***

In [1]:
import pandas as pd
#from treattech import treattech  # Assuming treattech is the function name in treattech.ipynb


def treatmentsteps(treatmentprocess, treatalt):
    treatmenttechnologies = []
    for treatstage in treatmentprocess:
        %run "/content/drive/My Drive/Colab Notebooks/WP6/treattech.ipynb"
        treatmenttechnologies.append(treattech(treatstage, treatmentprocess, treatalt))

    print("Treatment technologies selected:",treatmenttechnologies)
    treatmenttechnologies = pd.DataFrame(treatmenttechnologies, columns=['Treatment Technologies'])
    Treatment = pd.DataFrame({'Treatment Steps': treatmentprocess})

    Treatment = pd.concat([Treatment, treatmenttechnologies], axis=1)

    UF = list(Treatment[Treatment['Treatment Steps'].str.contains("Ultrafiltration")].index)
    OR = list(Treatment[Treatment['Treatment Steps'].str.contains("OMP Removal")].index)
    tt = Treatment.loc[OR, 'Treatment Technologies']

    if len(UF) == 0:
        SF = list(Treatment[Treatment['Treatment Steps'].str.contains("Sand Filtration")].index)
        dd = (tt != "Granular Activated Carbon Filtration (GAC)") & (tt != "NF/RO")
        if any(dd):
            temp_row = Treatment.loc[SF].copy()
            Treatment.iloc[SF] = Treatment.iloc[OR]
            Treatment.iloc[OR] = temp_row
    elif len(UF) > 0:
            dd = (tt == "Powdered Activated Carbon Dosing (PAC)")
            if any(dd):
                temp_row = Treatment.loc[UF].copy()
                Treatment.iloc[UF] = Treatment.iloc[OR]
                Treatment.iloc[OR] = temp_row

    FR = list(Treatment[Treatment['Treatment Steps'].str.contains("Floc Removal")].index)
    PAC = list(Treatment[Treatment['Treatment Technologies'].str.lower().str.strip().str.contains("powdered activated carbon dosing \(pac\)", case=False)].index)
    if len(FR)>0 and len(PAC)>0:
          temp_row = Treatment.iloc[PAC[0]].copy()
          Treatment.drop(PAC, inplace=True)
          temp_table = Treatment.iloc[FR[0]:]
          Treatment = Treatment.iloc[:FR[0]]
          Treatment = pd.concat([Treatment.iloc[:FR[0]], pd.DataFrame([temp_row]), temp_table], ignore_index=True)

    OZ = list(Treatment[Treatment['Treatment Technologies'].str.contains("Ozonation")].index)
    OR = list(Treatment[Treatment['Treatment Steps'].str.contains("OMP Removal")].index)
    if len(OR)>0 and len(OZ)>0 and OR==OZ:
#       temp_row = Treatment.iloc[DS[0]].copy()
#       Treatment.drop(index=DS, inplace=True)
       temp_table = Treatment.iloc[OR[0]:]
       Treatment = Treatment.iloc[:OR[0]-1]
       Treatment = pd.concat([Treatment, temp_table])
       dis = input( "Is additional disinfection required? (Yes/No)")
       if dis == "Yes":
          list_options = ["LP-UV", "Chlorination", "MF/UF"]
          idx = int(input( "Select one disinfection option: (0 for LP-UV, 1 for Chlorination, 2 for Chloramination)"))
          new_row = pd.DataFrame([{'Treatment Technologies': list_options[idx], 'Treatment Steps': "Additional disinfection"}])
          Treatment = pd.concat([Treatment, new_row], ignore_index=True)

    if (Treatment['Treatment Technologies'] == "Enhanced Coagulation").any():
       COAG = list(Treatment[Treatment['Treatment Steps'].str.contains("Coagulation")].index)
       COAG2 = list(Treatment[Treatment['Treatment Technologies'].str.contains("Enhanced Coagulation")].index)
       Treatment.loc[COAG, 'Treatment Steps'] = "Enhanced Coagulation"
       Treatment.drop(COAG2, inplace=True)
    elif (Treatment['Treatment Technologies'] == "Anionic IEX").any():
         pHcor = int(input("Is pH required? (1 for Yes and 2 for No)"))
         if pHcor == 1:
            idx = list(Treatment[Treatment['Treatment Steps'].str.contains("Disinfection")].index)
            if len(idx)>0:
               temp_table = Treatment.iloc[idx[0]:]
               Treatment = Treatment.iloc[:idx[0]-1]
               new_row = pd.DataFrame([{"Treatment Steps": "pH Correction", "Treatment Technologies": "pH Correction"}])
               Treatment = pd.concat([Treatment, new_row, temp_table], ignore_index=True)


    return Treatment